In [0]:
-- ---------- AGGREGATES ----------


In [0]:
-- Team consistency metrics (unchanged)
CREATE OR REPLACE MATERIALIZED VIEW agg_consistency_metrics AS
SELECT league_id, season, roster_id,
       stddev_pop(points_for) AS points_for_stddev,
       avg(points_for)       AS points_for_avg,
       count(*)              AS games_played,
       CASE WHEN avg(points_for) <> 0 THEN stddev_pop(points_for)/avg(points_for) END AS points_for_cv
FROM fact_team_week
GROUP BY league_id, season, roster_id;


In [0]:
-- Head to head records (unchanged)
CREATE OR REPLACE MATERIALIZED VIEW agg_rivalry_head_to_head AS
WITH a AS (SELECT * FROM fact_team_week),
     b AS (SELECT * FROM fact_team_week)
SELECT a.league_id, a.season,
       a.roster_id AS roster_id_a, b.roster_id AS roster_id_b,
       count(*) AS games_played,
       sum(CASE WHEN a.points_for > b.points_for THEN 1 ELSE 0 END) AS wins_a,
       sum(CASE WHEN a.points_for < b.points_for THEN 1 ELSE 0 END) AS wins_b,
       sum(CASE WHEN a.points_for = b.points_for THEN 1 ELSE 0 END) AS ties,
       avg(a.points_for) AS pf_per_game_a,
       avg(b.points_for) AS pf_per_game_b
FROM a JOIN b
  ON a.league_id=b.league_id AND a.season=b.season AND a.week=b.week
 AND a.roster_id<>b.roster_id
GROUP BY a.league_id, a.season, a.roster_id, b.roster_id;


In [0]:
-- All-time records (unchanged)
CREATE OR REPLACE MATERIALIZED VIEW agg_records_all_time AS
SELECT league_id, roster_id,
       max(points_for) AS max_points_for,
       min(points_for) AS min_points_for
FROM fact_team_week
GROUP BY league_id, roster_id;


In [0]:
-- Player total points per roster (answers "most points scored for team")
CREATE OR REPLACE MATERIALIZED VIEW agg_player_roster_totals AS
SELECT 
  pw.league_id,
  pw.roster_id,
  pw.player_id,
  p.full_name,
  p.position,
  min(pw.season) AS first_season,
  max(pw.season) AS last_season,
  count(DISTINCT pw.season) AS seasons_count,
  count(*) AS weeks_count,
  sum(pw.points) AS total_points,
  avg(pw.points) AS points_per_week,
  sum(CASE WHEN pw.was_started THEN pw.points ELSE 0 END) AS points_as_starter
FROM fact_player_week pw
LEFT JOIN dim_players p ON pw.player_id = p.player_id
GROUP BY pw.league_id, pw.roster_id, pw.player_id, p.full_name, p.position;

In [0]:
-- Draft ROI by round (enhanced with career tracking)
CREATE OR REPLACE MATERIALIZED VIEW agg_draft_roi_by_round AS
WITH picks AS (
  SELECT
    p.league_id,
    li.season,
    CAST(p.round AS INT) AS round,
    p.player_id
  FROM workspace.sleeper_raw.sleeper_draft_picks_snapshot p
  JOIN workspace.sleeper_raw.sleeper_league_info_snapshot li USING (league_id)
),
player_season_points AS (
  SELECT
    league_id, season, player_id,
    SUM(points) AS season_points
  FROM fact_player_week
  GROUP BY league_id, season, player_id
),
player_career_points AS (
  SELECT
    league_id, player_id,
    SUM(points) AS career_points
  FROM fact_player_week
  GROUP BY league_id, player_id
),
rook AS (
  SELECT
    pk.league_id, pk.season, pk.round, pk.player_id,
    COALESCE(psp.season_points, 0.0) AS rookie_season_points,
    COALESCE(pcp.career_points, 0.0) AS career_points
  FROM picks pk
  LEFT JOIN player_season_points psp
    ON pk.league_id = psp.league_id
   AND pk.season    = psp.season
   AND pk.player_id = psp.player_id
  LEFT JOIN player_career_points pcp
    ON pk.league_id = pcp.league_id
   AND pk.player_id = pcp.player_id
),
by_round AS (
  SELECT
    league_id, season, round,
    AVG(rookie_season_points) AS avg_rookie_points,
    AVG(career_points) AS avg_career_points,
    COUNT(*) AS picks_count
  FROM rook
  GROUP BY league_id, season, round
),
season_avg AS (
  SELECT
    league_id, season,
    AVG(rookie_season_points) AS season_avg_points,
    AVG(career_points) AS career_avg_points
  FROM rook
  GROUP BY league_id, season
)
SELECT
  b.league_id,
  b.season,
  b.round,
  b.picks_count,
  b.avg_rookie_points,
  b.avg_career_points,
  s.season_avg_points,
  s.career_avg_points,
  CASE WHEN s.season_avg_points > 0
       THEN b.avg_rookie_points / s.season_avg_points
       ELSE NULL
  END AS rookie_roi,
  CASE WHEN s.career_avg_points > 0
       THEN b.avg_career_points / s.career_avg_points
       ELSE NULL
  END AS career_roi
FROM by_round b
JOIN season_avg s
  ON b.league_id = s.league_id AND b.season = s.season;


In [ ]:
-- Weekly Value Over Replacement (VOR)
-- Calculates replacement level as the worst starter at each position per week
-- For 12-team leagues: QB12, RB24, WR36, TE12
CREATE OR REPLACE MATERIALIZED VIEW agg_player_week_vor AS
WITH league_sizes AS (
  SELECT 
    league_id,
    season,
    COUNT(DISTINCT roster_id) AS num_teams
  FROM fact_team_week
  GROUP BY league_id, season
),
position_replacement_thresholds AS (
  SELECT 
    ls.league_id,
    ls.season,
    ls.num_teams,
    -- Standard starting lineup: 1 QB, 2 RB, 3 WR, 1 TE, 1 FLEX (assumes RB/WR/TE eligible)
    ls.num_teams * 1 AS qb_replacement_rank,
    ls.num_teams * 2 AS rb_replacement_rank,
    ls.num_teams * 3 AS wr_replacement_rank,
    ls.num_teams * 1 AS te_replacement_rank
  FROM league_sizes ls
),
weekly_position_rankings AS (
  SELECT
    pw.league_id,
    pw.season,
    pw.week,
    pw.player_id,
    pw.position,
    pw.points,
    pw.was_started,
    ROW_NUMBER() OVER (
      PARTITION BY pw.league_id, pw.season, pw.week, pw.position 
      ORDER BY pw.points DESC
    ) AS position_rank
  FROM fact_player_week pw
  WHERE pw.position IN ('QB', 'RB', 'WR', 'TE')
),
replacement_levels AS (
  SELECT
    wpr.league_id,
    wpr.season,
    wpr.week,
    wpr.position,
    CASE 
      WHEN wpr.position = 'QB' THEN prt.qb_replacement_rank
      WHEN wpr.position = 'RB' THEN prt.rb_replacement_rank
      WHEN wpr.position = 'WR' THEN prt.wr_replacement_rank
      WHEN wpr.position = 'TE' THEN prt.te_replacement_rank
    END AS replacement_rank,
    MAX(CASE 
      WHEN wpr.position = 'QB' AND wpr.position_rank = prt.qb_replacement_rank THEN wpr.points
      WHEN wpr.position = 'RB' AND wpr.position_rank = prt.rb_replacement_rank THEN wpr.points
      WHEN wpr.position = 'WR' AND wpr.position_rank = prt.wr_replacement_rank THEN wpr.points
      WHEN wpr.position = 'TE' AND wpr.position_rank = prt.te_replacement_rank THEN wpr.points
    END) AS replacement_points
  FROM weekly_position_rankings wpr
  JOIN position_replacement_thresholds prt 
    ON wpr.league_id = prt.league_id AND wpr.season = prt.season
  GROUP BY wpr.league_id, wpr.season, wpr.week, wpr.position, prt.qb_replacement_rank, prt.rb_replacement_rank, prt.wr_replacement_rank, prt.te_replacement_rank
)
SELECT
  wpr.league_id,
  wpr.season,
  wpr.week,
  wpr.player_id,
  p.full_name,
  wpr.position,
  wpr.points,
  wpr.was_started,
  wpr.position_rank,
  COALESCE(rl.replacement_points, 0.0) AS replacement_level_points,
  wpr.points - COALESCE(rl.replacement_points, 0.0) AS vor,
  CASE 
    WHEN COALESCE(rl.replacement_points, 0.0) > 0 
    THEN (wpr.points - COALESCE(rl.replacement_points, 0.0)) / rl.replacement_points 
    ELSE NULL 
  END AS vor_percentage
FROM weekly_position_rankings wpr
LEFT JOIN replacement_levels rl 
  ON wpr.league_id = rl.league_id 
  AND wpr.season = rl.season 
  AND wpr.week = rl.week 
  AND wpr.position = rl.position
LEFT JOIN dim_players p ON wpr.player_id = p.player_id;

In [ ]:
-- Seasonal player stats across all rosters
-- Aggregates player performance per season across the entire league
CREATE OR REPLACE MATERIALIZED VIEW agg_player_season_stats AS
WITH player_weekly_stats AS (
  SELECT
    pw.league_id,
    pw.season,
    pw.player_id,
    p.full_name,
    p.position,
    COUNT(*) AS weeks_rostered,
    SUM(CASE WHEN pw.was_started THEN 1 ELSE 0 END) AS weeks_started,
    SUM(pw.points) AS total_points,
    SUM(CASE WHEN pw.was_started THEN pw.points ELSE 0 END) AS points_when_started,
    SUM(CASE WHEN NOT pw.was_started THEN pw.points ELSE 0 END) AS points_when_benched,
    AVG(pw.points) AS avg_points_per_week,
    AVG(CASE WHEN pw.was_started THEN pw.points ELSE NULL END) AS avg_points_when_started,
    AVG(CASE WHEN NOT pw.was_started THEN pw.points ELSE NULL END) AS avg_points_when_benched,
    MAX(pw.points) AS max_weekly_points,
    MIN(pw.points) AS min_weekly_points,
    STDDEV_POP(pw.points) AS points_stddev
  FROM fact_player_week pw
  LEFT JOIN dim_players p ON pw.player_id = p.player_id
  GROUP BY pw.league_id, pw.season, pw.player_id, p.full_name, p.position
),
player_vor_stats AS (
  SELECT
    vor.league_id,
    vor.season,
    vor.player_id,
    SUM(vor.vor) AS total_vor,
    AVG(vor.vor) AS avg_weekly_vor,
    SUM(CASE WHEN vor.was_started THEN vor.vor ELSE 0 END) AS vor_when_started,
    MAX(vor.vor) AS max_weekly_vor,
    MIN(vor.vor) AS min_weekly_vor
  FROM agg_player_week_vor vor
  GROUP BY vor.league_id, vor.season, vor.player_id
)
SELECT
  pws.league_id,
  pws.season,
  pws.player_id,
  pws.full_name,
  pws.position,
  pws.weeks_rostered,
  pws.weeks_started,
  pws.total_points,
  pws.points_when_started,
  pws.points_when_benched,
  pws.avg_points_per_week,
  pws.avg_points_when_started,
  pws.avg_points_when_benched,
  pws.max_weekly_points,
  pws.min_weekly_points,
  pws.points_stddev,
  COALESCE(pvs.total_vor, 0.0) AS total_vor,
  COALESCE(pvs.avg_weekly_vor, 0.0) AS avg_weekly_vor,
  COALESCE(pvs.vor_when_started, 0.0) AS vor_when_started,
  COALESCE(pvs.max_weekly_vor, 0.0) AS max_weekly_vor,
  COALESCE(pvs.min_weekly_vor, 0.0) AS min_weekly_vor,
  CASE 
    WHEN pws.weeks_rostered > 0 
    THEN CAST(pws.weeks_started AS DOUBLE) / CAST(pws.weeks_rostered AS DOUBLE) 
    ELSE 0.0 
  END AS start_rate
FROM player_weekly_stats pws
LEFT JOIN player_vor_stats pvs 
  ON pws.league_id = pvs.league_id 
  AND pws.season = pvs.season 
  AND pws.player_id = pvs.player_id;